# Using Prompt for Base Model

In [1]:
# dependencies
%pip install datasets -q transformers accelerate torch

In [2]:
# Preview Datasets
from datasets import load_dataset

# Generate JSON Output
import json

# Load model
import torch
import re
from transformers import pipeline

In [3]:
# Configurations

# Dataset
DATASETS = {
    "gsm8k": ("gsm8k", "main"),
    # "asdiv": ("yimingzhang/asdiv", None),
    # "metamathqa": ("meta-math/MetaMathQA", None),
    # "openmathinstruct": ("nvidia/OpenMathInstruct-1", None), # Can change to nvidia/OpenMathInstruct-2 for larger datasets
}

DATASET_CONFIG = {
    "gsm8k": {
        "path": "gsm8k",
        "subset": "main",
        "split": "test",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["answer"]
    }#,
    # "asdiv": {
    #     "path": "yimingzhang/asdiv",
    #     "subset": None,
    #     "split": "test",
    #     "extract_question": lambda x: x["text"].replace("Question:", "").replace("Answer:", "").strip(),
    #     "extract_answer": lambda x: x["label"]   # clean numeric answer
    # },
    # "metamathqa": {
    #     "path": "meta-math/MetaMathQA",
    #     "subset": None,
    #     "split": "train", # No testing dataset available
    #     "extract_question": lambda x: x["original_question"],
    #     "extract_answer": lambda x: x["response"]
    # },
    # "openmathinstruct": {
    #     "path": "nvidia/OpenMathInstruct-1", # Can change to nvidia/OpenMathInstruct-2 for larger datasets
    #     "subset": None,
    #     "split": "validation",
    #     "extract_question": lambda x: x["question"],
    #     "extract_answer": lambda x: x["expected_answer"]
    }
#}

# Model
MODEL_NAME = "Qwen/Qwen2-1.5B"

DEVICE = 0 if torch.cuda.is_available() else -1

TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# Key Functions

# PROMPT
# Build Prompt
def build_prompt(question, dataset_name):
    return f"""
You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step using clear logical reasoning.
Show all intermediate reasoning clearly.
At the end, provide the final numerical answer.

Follow these steps:
1. Analyze the problem.
2. Compute step by step.
3. Double-check calculations.
4. Provide final result.

Important rules:
- Do NOT include any text outside the JSON.
- Do NOT use markdown.
- Ensure the final answer is a single number or expression.
- Do NOT restate the instructions.

Output your response strictly in the following JSON format:
{{
  "dataset": "{dataset_name}",
  "question": "...",
  "model_reasoning": "...",
  "model_answer": "..."
}}

Problem:
{question}
"""

def parse_model_output(raw_response):
    json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw_response)
    if json_match:
        json_str = json_match.group(0)
    else:
        start_idx = raw_response.find('{')
        end_idx = raw_response.rfind('}') + 1
        json_str = raw_response[start_idx:end_idx] if start_idx != -1 else '{}'

    try:
        model_output = json.loads(json_str)
    except json.JSONDecodeError:
        model_output = {"model_reasoning": "", "model_answer": raw_response}

    return model_output

# MODEL

# Get specific answer
def extract_number(text):
    numbers = re.findall(r"\d+", text)
    if numbers:
        return int(numbers[-1])
    return None

# Input prompt to model
def ask_model(prompt, dataset_name, question, answer, max_tokens=256):
    output = pipe(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    raw_response = output[0]["generated_text"].strip()
    predicted_number = extract_number(raw_response)
    raw_output = parse_model_output(raw_response)

    reasoning = raw_output.get("model_reasoning", "")
    model_answer = raw_output.get("model_answer", raw_response)
    
    # Combine reasoning and answer in the format: reasoning\n#### answer
    model_output = f"{reasoning}" + " " + f"{model_answer}"
    combined_answer = f"{model_output}" + " \n#### " + f"{predicted_number}"

    response = {
        "dataset": dataset_name,
        "question": question,
        "model_answer": combined_answer,
        "correct_answer": answer
    }

    return response

# def ask_model_batch(prompts, names, questions, answers, max_tokens=256, batch_size=8):
#     if not prompts:
#         return []

#     outputs = pipe(
#         prompts,
#         max_new_tokens=max_tokens,
#         do_sample=False,
#         temperature=0.0,
#         return_full_text=False,
#         batch_size=batch_size
#     )

#     responses = []
#     for output, name, question, answer in zip(outputs, names, questions, answers):
#         if isinstance(output, list):
#             generated_text = output[0].get("generated_text", "").strip()
#         else:
#             generated_text = output.get("generated_text", "").strip()

#         model_output = parse_model_output(generated_text)
#         responses.append({
#             "dataset": name,
#             "question": question,
#             "model_reasoning": model_output.get("model_reasoning", ""),
#             "model_answer": model_output.get("model_answer", generated_text),
#             "correct_answer": answer
#         })

#     return responses

def extract_number(text):
    numbers = re.findall(r"\d+", text)
    if numbers:
        return int(numbers[-1])
    return None


def evaluate_math_question(question, correct_answer):
    response = ask_model(question)
    predicted = extract_number(response)

    print("Question:", question)
    print("Model Response:", response)
    print("Extracted Answer:", predicted)
    print("Correct Answer:", correct_answer)

    if predicted == correct_answer:
        print("✅ Correct\n")
        return True
    else:
        print("❌ Incorrect\n")
        return False


In [5]:
for name, (path, subset) in DATASETS.items():
    print("\n" + "=" * 70)
    print(f"Dataset: {name}")
    print("=" * 70)

    try:
        # Load dataset
        if subset:
            dataset = load_dataset(path, subset)
        else:
            dataset = load_dataset(path)

        print("Available splits:", list(dataset.keys()))

        for split in dataset.keys():
            print(f"\n--- Split: {split} ---")
            print("Number of samples:", len(dataset[split]))

            # Column names
            print("Column names:", dataset[split].column_names)

            # Feature types (schema)
            print("Features:")
            print(dataset[split].features)

            # Print one example
            print("\nFirst sample:")
            print(dataset[split][0])

    except Exception as e:
        print("Error loading dataset:", e)


Dataset: gsm8k


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Available splits: ['train', 'test']

--- Split: train ---
Number of samples: 7473
Column names: ['question', 'answer']
Features:
{'question': Value('string'), 'answer': Value('string')}

First sample:
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

--- Split: test ---
Number of samples: 1319
Column names: ['question', 'answer']
Features:
{'question': Value('string'), 'answer': Value('string')}

First sample:
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 1

In [6]:
pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE
)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [25]:
all_outputs = []

for dataset_name, config in DATASET_CONFIG.items():

    print(f"Processing {dataset_name}...")

    dataset = load_dataset(
        config["path"],
        config["subset"] if config["subset"] else None
    )

    split_data = dataset[config["split"]]

    for sample in split_data:

        question = config["extract_question"](sample)
        answer = config.get("extract_answer", lambda x: None)(sample)        
        prompt = build_prompt(question, dataset_name)

        # ---- send to your model here ----
        # response_text = model.generate(prompt)
        response = ask_model(prompt, dataset_name, question, answer)

        # For demonstration only:
        # response = {
        #     "dataset": dataset,
        #     "question": question,
        #     "model_answer": model_answer,
        #     "expected_answer": answer
        # }

        all_outputs.append(response)

        # Optional: limit samples per dataset
        if len(all_outputs) >= 100:
            break

Processing gsm8k...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [26]:
len(all_outputs)

100

In [27]:
all_outputs[:5]

[{'dataset': 'gsm8k',
  'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'model_answer': "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. So, she has 16 - 3 - 4 = 9 eggs left to sell at the farmers' market. She sells each egg for $2, so she makes 9 * $2 = $18 every day at the farmers' market. $18 \n#### 8",
  'correct_answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'},
 {'dataset': 'gsm8k',
  'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
  'model_answer': 'A robe takes 2 bolts of blue fiber an

In [31]:
all_outputs[:-5]

[{'dataset': 'gsm8k',
  'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'model_answer': "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. So, she has 16 - 3 - 4 = 9 eggs left to sell at the farmers' market. She sells each egg for $2, so she makes 9 * $2 = $18 every day at the farmers' market. $18 \n#### 8",
  'correct_answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'},
 {'dataset': 'gsm8k',
  'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
  'model_answer': 'A robe takes 2 bolts of blue fiber an

In [42]:
import os
import shutil

# Copy from current dir to output/ (if exists)
if os.path.exists('all_outputs.json'):
    os.makedirs('output', exist_ok=True)
    shutil.copy2('all_outputs.json', 'output/all_outputs.json')
    print("Copied to output/all_outputs.json - ready for download!")
else:
    print("all_outputs.json not found in current dir. Save it first.")

# Verify
print("Output dir:", os.listdir('output/') if os.path.exists('output/') else 'empty')

Copied to output/all_outputs.json - ready for download!
Output dir: ['all_outputs.json']


In [30]:
# Save to JSON
with open("all_outputs.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

In [41]:
import json
import os
import shutil

# os.makedirs('math_output', exist_ok=True)
with open('math_output/all_outputs.json', 'w', encoding='utf-8') as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

print("Saved to math_output/all_outputs.json - check downloads!")

Saved to math_output/all_outputs.json - check downloads!


In [37]:
print("Current working directory:", os.getcwd())
print("\nContents of current directory:")
print(os.listdir('.'))

Current working directory: /content

Contents of current directory:
['.config', 'all_outputs.json', 'math_output', 'output', 'LLM-project', 'sample_data']


In [39]:
import json
import os
import shutil

# Define paths
current_path = 'all_outputs.json'
project_path = 'LLM-project/all_outputs.json'
output_path = 'output/all_outputs.json'

# Save to current directory
with open(current_path, "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

# Ensure LLM-project dir exists, then save there
os.makedirs('LLM-project', exist_ok=True)
with open(project_path, "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

# Copy to output/ for download
os.makedirs('output', exist_ok=True)
shutil.copy2(current_path, output_path)

print(f"Saved to: {current_path}, {project_path}, {output_path}")

Saved to: all_outputs.json, LLM-project/all_outputs.json, output/all_outputs.json


In [38]:
import os

# Ensure output directory exists
os.makedirs('/content/LLM-project/math_output/', exist_ok=True)

#Save to math_output/all_outputs.json
file_path = '/content/LLM-project/math_output/all_outputs.json'
try:
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(all_outputs, f, indent=4, ensure_ascii=False)
    print(f"Successfully saved to {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")
except Exception as e:
    print(f"Error: {e}")

file_path = '/content/LLM-project/math_output/all_outputs.json'
if os.path.exists(file_path):
    size = os.path.getsize(file_path)
    print(f"File exists, size: {size} bytes")
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Keys in JSON: {list(data.keys())}")
else:
    print("File not found")

Successfully saved to /content/LLM-project/math_output/all_outputs.json
File size: 107061 bytes
File exists, size: 107061 bytes


AttributeError: 'list' object has no attribute 'keys'

In [ ]:
# KIV: Don't use this for now as it takes a long time to run and we can just test on a few samples for now
all_outputs = []

MAX_SAMPLES_TOTAL = 1000
MAX_PIPELINE_CALLS_PER_DATASET = 2
PIPELINE_BATCH_SIZE = 8

for dataset, config in DATASET_CONFIG.items():
    if len(all_outputs) >= MAX_SAMPLES_TOTAL:
        break

    print(f"Processing {dataset}...")

    dataset = load_dataset(
        config["path"],
        config["subset"] if config["subset"] else None
    )

    split_data = dataset[config["split"]]

    remaining = MAX_SAMPLES_TOTAL - len(all_outputs)
    dataset_records = []

    for sample in split_data:
        question = config["extract_question"](sample)
        answer = config.get("extract_answer", lambda x: None)(sample)
        prompt = build_prompt(question, dataset)
        dataset_records.append((prompt, dataset, question, answer))

        if len(dataset_records) >= remaining:
            break

    if not dataset_records:
        continue

    num_calls = min(MAX_PIPELINE_CALLS_PER_DATASET, len(dataset_records))
    chunk_size = (len(dataset_records) + num_calls - 1) // num_calls

    for start in range(0, len(dataset_records), chunk_size):
        chunk = dataset_records[start:start + chunk_size]
        prompts = [record[0] for record in chunk]
        names = [record[1] for record in chunk]
        questions = [record[2] for record in chunk]
        answers = [record[3] for record in chunk]

        responses = ask_model_batch(
            prompts,
            names,
            questions,
            answers,
            batch_size=PIPELINE_BATCH_SIZE
        )

        all_outputs.extend(responses)

NameError: name 'DATASET_CONFIG' is not defined

In [2]:
# Save to JSON
with open("all_outputs.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

NameError: name 'json' is not defined

# Past Test Cases

In [21]:
# MAIN TEST CASE
question = "If I have 5 red apples and 7 green apples, how many apples do I have?"

prompt = build_prompt(question, "gsm8k")

response = ask_model(prompt, "gsm8k", question, "12") # Providing correct answer for evaluation

print("Model Output:")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model Output:
{'dataset': 'gsm8k', 'question': 'If I have 5 red apples and 7 green apples, how many apples do I have?', 'model_answer': 'To find the total number of apples, we can add the number of red apples to the number of green apples. There are 12 apples in total. \n#### 12', 'correct_answer': '12'}


In [8]:
print(type(response))

<class 'dict'>


In [9]:
print(prompt)


You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step using clear logical reasoning.
Show all intermediate reasoning clearly.
At the end, provide the final numerical answer.

Follow these steps:
1. Analyze the problem.
2. Compute step by step.
3. Double-check calculations.
4. Provide final result.

Important rules:
- Do NOT include any text outside the JSON.
- Do NOT use markdown.
- Ensure the final answer is a single number or expression.
- Do NOT restate the instructions.

Output your response strictly in the following JSON format:
{
  "dataset": "gsm8k",
  "question": "...",
  "model_reasoning": "...",
  "model_answer": "..."
}

Problem:
If I have 5 red apples and 7 green apples, how many apples do I have?



In [ ]:
# Test Case 1
question = "If I have 5 red apples and 7 green apples, how many apples do I have?"

response = ask_model(question)

print("Model Output:")
print(response)

In [10]:
questions = [
    ("If I have 5 red apples and 7 green apples, how many apples do I have?", 12),
    ("John has 10 books and buys 3 more. How many books does he have?", 13),
    ("There are 20 birds, 5 fly away. How many remain?", 15),
    ("Sarah has 8 candies and eats 2. How many are left?", 6),
]

correct = 0

for q, ans in questions:
    if evaluate_math_question(q, ans):
        correct += 1

accuracy = (correct / len(questions))*100
print(f"Final Accuracy: {accuracy:.2f}")

TypeError: ask_model() missing 3 required positional arguments: 'name', 'question', and 'answer'